# SegFormer B0/B2/B5 - Olcek ablasyonu

**Katman:** Guclendirici

Olcek-performans egrisi: uc cihaz argumani icin Params/GFLOPs ekseni. Ayni protokolde uc kodlayici boyutu.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/01_yol_hasari
    %pip install -q segmentation-models-pytorch albumentations roboflow thop opencv-python-headless
print('cwd:', os.getcwd())

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import seg_common as SC, torch
SC.CONFIG.update(dict(roboflow_api_key='BURAYA', roboflow_workspace='w', roboflow_project='p', roboflow_version=1))
paths = SC.prepare_dataset(); dl = SC.build_loaders(paths)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
for enc in ['mit_b0', 'mit_b2', 'mit_b5']:
    name = f'segformer_{enc}'
    m = SC.build_model('Segformer', enc, SC.CONFIG['num_classes'])
    m = SC.train_model(m, dl, SC.CONFIG)
    split = 'test' if 'test' in dl else 'valid'
    met = SC.evaluate_seg(m, dl[split], dev)
    eff = SC.efficiency_stats(m, SC.CONFIG['img_size'], 'cpu')
    SC.save_result(name, met, eff, extra={'encoder': enc})
    print(enc, met['mean_iou'], eff)